# 01 - Exploratory Data Analysis (EDA)

Bitcoin Price Prediction Project

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
df = pd.read_csv('../archive/btcusd_1-min_data.csv')
df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='s')
df = df.sort_values('Timestamp').reset_index(drop=True)
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['Timestamp'].min()} to {df['Timestamp'].max()}")

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print(f"\nDuplicates: {df.duplicated().sum()}")

In [ ]:
df_daily = df.set_index('Timestamp').resample('1D').agg({
    'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'
}).dropna()

fig = px.line(df_daily, x=df_daily.index, y='Close', title='BTC Closing Price Over Time')
fig.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(df_daily['Close'])
axes[0, 0].set_title('Close Price')
axes[0, 0].set_ylabel('USD')

axes[0, 1].plot(df_daily['Volume'])
axes[0, 1].set_title('Volume')
axes[0, 1].set_ylabel('Volume')

axes[1, 0].hist(df_daily['Close'], bins=50, edgecolor='black')
axes[1, 0].set_title('Price Distribution')

df_daily['returns'] = df_daily['Close'].pct_change()
axes[1, 1].hist(df_daily['returns'].dropna(), bins=50, edgecolor='black')
axes[1, 1].set_title('Returns Distribution')

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(df_daily['Close'].dropna(), lags=50, ax=axes[0])
plot_pacf(df_daily['Close'].dropna(), lags=50, ax=axes[1])
plt.show()

In [ ]:
df_daily['rolling_mean_30'] = df_daily['Close'].rolling(window=30).mean()
df_daily['rolling_std_30'] = df_daily['Close'].rolling(window=30).std()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df_daily['Close'], label='Close', alpha=0.7)
ax.plot(df_daily['rolling_mean_30'], label='30-day MA', color='orange')
ax.fill_between(df_daily.index, 
                df_daily['rolling_mean_30'] - 2*df_daily['rolling_std_30'],
                df_daily['rolling_mean_30'] + 2*df_daily['rolling_std_30'],
                alpha=0.2, label='±2 std')
ax.legend()
ax.set_title('BTC Price with Moving Average')
plt.show()

In [ ]:
df_daily['year'] = df_daily.index.year
yearly_stats = df_daily.groupby('year')['Close'].agg(['mean', 'min', 'max', 'std'])
print(yearly_stats)

## Key Findings

- Dataset contains ~7.5M minute-level records
- Date range: early 2012 to late 2021
- Strong upward trend with high volatility
- Returns show fat tails (non-normal distribution)
- High autocorrelation (typical for price series)